In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
import torchvision.models as models

from sklearn.metrics import accuracy_score, classification_report

# Loading and Preprocessing

In [68]:
# Transform image: Min max scaling (0, 1), Normalize (-1, 1)
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)), # increasing size to match alex net 

        # data augmentaiton 
        transforms.RandomHorizontalFlip(), # flip horizontally
        transforms.ColorJitter(brightness=0.2), # 
        transforms.RandomResizedCrop(size=224, scale=(0.8, 1.0), antialias=True), # randomly crop keeping 80-100%

        transforms.ToTensor(), # convert to tensore and min max scale
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), # Normalize (-1, 1)
    ]
)

In [69]:
# Loading dataset
trainset = CIFAR10(
    root="datasets/cnn_data", 
    # for first time it was True
    download=False,
    transform=transform,
    train=True
)
testset = CIFAR10(
    root="datasets/cnn_data",
    transform=transform,
    train=False
)

In [70]:
trainset, valset = random_split(
    dataset=trainset,
    lengths=[0.8, 0.2]
)

In [71]:
# Convert to DataLoader
# CIFAR10 already returned the datasets
train_loader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    valset,
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    testset,
    batch_size=64,
    shuffle=True
)

# Loading model

In [72]:
# loading the model with pre trained weights
rn_model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

In [73]:
rn_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [74]:
# # freezing current layers
# for param in an_model.parameters():
#     param.requires_grad = False

In [75]:
# appending extra layers at end to make output compatible with our labels
rn_model.fc = nn.Linear(512, 10)

In [76]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(rn_model.parameters())

# Training model

In [77]:
# Training
epochs = 10
min_val_loss = float("inf")

for epoch in range(0, epochs):
    # training
    rn_model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = rn_model.forward(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    # validation
    rn_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            y_pred = rn_model.forward(images)
            loss = criterion(y_pred, labels)

            val_loss += loss.item()

    val_loss /= len(test_loader)
    if (val_loss < min_val_loss):
        min_val_loss = val_loss
        torch.save(rn_model.state_dict(), "cifar_10_cnn.pt")


    print(f"Epoch: {epoch} Train => {train_loss} Val => {val_loss}")


KeyboardInterrupt: 

# Evaluation

In [ ]:
# Evaluation
rn_model.eval()

y_actual = []
y_pred = []

with torch.no_grad():
    for xb, yb in test_loader:
        y_pred_probs = rn_model(xb)
        _, y_pred_labels = torch.max(y_pred_probs, 1)

        y_actual.extend(yb.cpu().numpy())
        y_pred.extend(y_pred_labels.cpu().numpy())


In [ ]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(f"Classification report: {classification_report(y_actual, y_pred)}")